# Solutions · 11 · Large-amplitude oscillatory shear (LAOS)

Worked solutions to the exercises of [notebook 11](../notebooks/11_laos.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engrheo is missing): install it from GitHub
    import engrheo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engrheo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engrheo import datasets, laos, oscillatory, constitutive as cm
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
yo = pd.read_csv(datasets.path("yoghurt_laos.csv"), comment="#")
w_y = 2 * np.pi

## Exercise 1

Simulate LAOS of the UCM model (the same spectrum, no Giesekus term) at strain 20. How large is $I_3/I_1$, and why is the UCM shear stress linear in strain at any amplitude?

In [3]:
ucm = cm.modes_from_spectrum([60.0, 25.0, 8.0], [0.02, 0.2, 2.0])
r = cm.laos_shear(ucm, 20.0, 2.0, n_cycles=12)
print(f"UCM at strain amplitude 20: I3/I1 = {laos.harmonics(r['t'][:-1], r['strain'][:-1], r['stress'][:-1], 2.0)['I3_I1']:.2e}")

UCM at strain amplitude 20: I3/I1 = 2.91e-10


Essentially zero. In simple shear the UCM equation for the shear stress, $\lambda\dot\tau_{xy} = -\tau_{xy} + \eta\dot\gamma + \lambda\dot\gamma\tau_{yy}$,
contains the normal stress $\tau_{yy}$ - which is always zero for UCM. The shear stress therefore obeys a *linear*
equation at any amplitude; the nonlinearity appears only in the normal stresses. Shear-stress harmonics
need a model such as Giesekus or PTT.

## Exercise 2

For the yoghurt, at which strain amplitude does $G'_L$ fall below half of its small-strain value? Compare with the flow point from the first-harmonic moduli (`oscillatory.moduli_from_waveforms` for each amplitude).

In [4]:
rows = []
for g0, grp in yo.groupby("strain_amplitude"):
    c = laos.chebyshev(grp.time_s, grp.strain, grp.shear_stress_Pa, w_y)
    m = oscillatory.moduli_from_waveforms(grp.time_s, grp.strain, grp.shear_stress_Pa, w_y)
    rows.append((g0, c["G_L"], m["Gp"], m["Gpp"]))
t = pd.DataFrame(rows, columns=["strain_amp", "G'_L", "G'_1", "G''_1"])
print(t.to_string(index=False, formatters={"strain_amp": "{:g}".format, "G'_L": "{:.1f}".format,
                                           "G'_1": "{:.1f}".format, "G''_1": "{:.1f}".format}))
lin = t["G'_L"].iloc[0]
below = t[t["G'_L"] < lin / 2]
print(f"G'_L falls below half of its linear value ({lin/2:.0f} Pa) first at strain amplitude {below.strain_amp.iloc[0]:g}")
print(f"first-harmonic G' > G'' at every amplitude tested: {bool(np.all(t['G' + chr(39) + '_1'] > t['G' + chr(39)*2 + '_1']))}")

strain_amp  G'_L  G'_1 G''_1
     0.005 300.1 300.9 107.1
      0.02 311.6 308.1  90.2
      0.08 432.4 406.5  66.3
       0.3 291.9 347.9  41.9
         1  72.6  93.8  25.1
G'_L falls below half of its linear value (150 Pa) first at strain amplitude 1
first-harmonic G' > G'' at every amplitude tested: True


At 100 % strain the large-strain modulus has collapsed to about a quarter of its linear value - the gel yields at
the extremes of every cycle - yet the first-harmonic moduli still report $G' > G''$, a "solid", at every amplitude
tested: no flow point is reached. Averaged over the cycle, the stiff behaviour near zero strain masks the yielding
at large strain. The Chebyshev measures resolve that intra-cycle yielding; the first harmonic alone would miss it.

## Exercise 3

**Implement it yourself:** the area of the elastic Lissajous loop equals the energy dissipated per cycle, $\pi\gamma_0^2 G''_1$. Compute the loop area of one yoghurt cycle with the shoelace formula and compare.

In [5]:
g0, grp = list(yo.groupby("strain_amplitude"))[2]
x, y = grp.strain.to_numpy(), grp.shear_stress_Pa.to_numpy()
area = 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))
Gpp1 = oscillatory.moduli_from_waveforms(grp.time_s, x, y, w_y)["Gpp"]
print(f"strain {g0}: loop area {area:.4f} J/m3, pi gamma0^2 G''_1 = {np.pi * g0**2 * Gpp1:.4f} J/m3")

strain 0.08: loop area 1.3336 J/m3, pi gamma0^2 G''_1 = 1.3338 J/m3


The area of the elastic Lissajous loop is the energy dissipated per unit volume and cycle. Only the first
harmonic contributes to it (higher odd harmonics integrate to zero over a cycle), so it equals $\pi\gamma_0^2 G''_1$
even in the nonlinear regime - up to the discretisation of the sampled loop.